# 06 — Twitch Ranking Evaluation

Compare all trained rankers on the same held-out candidate sets and inspect important failure modes:

- overall discrimination
- Recall / NDCG / MRR
- familiar vs new-streamer positives
- popularity bias
- example recommendation lists

This notebook consumes predictions saved by Notebooks 03 and 04.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_DIR = Path("Distributed ML Inference & Ranking")
DATA_DIR = PROJECT_DIR / "data/processed/twitch"
PRED_DIR = DATA_DIR / "predictions"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/twitch"

TEST_PATH = DATA_DIR / "twitch_test_ranking.csv.gz"
BASELINE_PRED_PATH = PRED_DIR / "baseline_predictions.csv.gz"
NEURAL_PRED_PATH = PRED_DIR / "neural_predictions.csv.gz"

test = pd.read_csv(TEST_PATH)
baseline = pd.read_csv(BASELINE_PRED_PATH)
neural = pd.read_csv(NEURAL_PRED_PATH)

pred = baseline.merge(
    neural[
        ["user_id", "streamer", "score_neural"]
    ],
    on=["user_id", "streamer"],
    how="inner",
    validate="one_to_one",
)

print("Predictions:", pred.shape)
pred.head()

## 1. Metric functions

In [ ]:
def evaluate_score(frame, score_col, ks=(5, 10, 20)):
    ranked = frame.sort_values(
        ["user_id", score_col],
        ascending=[True, False],
    )

    rows = []

    for user_id, group in ranked.groupby("user_id", sort=False):
        labels = group["label"].to_numpy()
        positives = labels.sum()

        if positives == 0:
            continue

        row = {"user_id": user_id}
        positions = np.flatnonzero(labels == 1)
        row["MRR"] = 1.0 / (positions[0] + 1)

        for k in ks:
            top = labels[:k]
            hits = top.sum()

            row[f"Recall@{k}"] = hits / positives
            row[f"HitRate@{k}"] = float(hits > 0)

            discounts = 1 / np.log2(
                np.arange(2, len(top) + 2)
            )
            dcg = (top * discounts).sum()

            ideal_len = min(int(positives), k)
            idcg = (
                1 / np.log2(
                    np.arange(2, ideal_len + 2)
                )
            ).sum()

            row[f"NDCG@{k}"] = dcg / idcg

        rows.append(row)

    per_user = pd.DataFrame(rows)

    summary = {
        "model": score_col,
        "ROC_AUC": roc_auc_score(
            frame["label"],
            frame[score_col],
        ),
        "AveragePrecision": average_precision_score(
            frame["label"],
            frame[score_col],
        ),
        "MRR": per_user["MRR"].mean(),
    }

    for k in ks:
        summary[f"Recall@{k}"] = per_user[f"Recall@{k}"].mean()
        summary[f"HitRate@{k}"] = per_user[f"HitRate@{k}"].mean()
        summary[f"NDCG@{k}"] = per_user[f"NDCG@{k}"].mean()

    return summary, per_user

## 2. Overall model comparison

In [ ]:
SCORE_COLUMNS = [
    "score_popularity",
    "score_logreg",
    "score_hgb",
    "score_neural",
]

summaries = []
per_user_results = {}

for score_col in SCORE_COLUMNS:
    summary, per_user = evaluate_score(
        pred,
        score_col,
    )

    summaries.append(summary)
    per_user_results[score_col] = per_user

comparison = (
    pd.DataFrame(summaries)
    .sort_values("NDCG@10", ascending=False)
    .reset_index(drop=True)
)

comparison

## 3. Plot NDCG@10

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    comparison["model"],
    comparison["NDCG@10"],
)

plt.ylabel("NDCG@10")
plt.title("Ranking Quality by Model")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 4. Familiar vs new-streamer positives

In [ ]:
analysis = pred.merge(
    test[
        [
            "user_id",
            "streamer",
            "pair_interactions",
            "pair_watch_share",
            "streamer_interactions",
            "streamer_popularity_share",
        ]
    ],
    on=["user_id", "streamer"],
    how="left",
    validate="one_to_one",
)

positive_analysis = analysis[
    analysis["label"] == 1
].copy()

positive_analysis["positive_type"] = np.where(
    positive_analysis["seen_before"] == 1,
    "Repeat streamer",
    "New streamer",
)

positive_analysis["positive_type"].value_counts(normalize=True)

## 5. Score separation by positive type

In [ ]:
segment_rows = []

for model_col in SCORE_COLUMNS:
    for segment, group in positive_analysis.groupby("positive_type"):
        segment_rows.append(
            {
                "model": model_col,
                "segment": segment,
                "mean_score": group[model_col].mean(),
                "median_score": group[model_col].median(),
                "count": len(group),
            }
        )

pd.DataFrame(segment_rows)

## 6. Popularity bias in Top-K recommendations

In [ ]:
def topk_popularity(frame, score_col, k=10):
    top = (
        frame.sort_values(
            ["user_id", score_col],
            ascending=[True, False],
        )
        .groupby("user_id")
        .head(k)
    )

    return {
        "model": score_col,
        "mean_streamer_interactions": (
            top["streamer_interactions"].mean()
        ),
        "median_streamer_interactions": (
            top["streamer_interactions"].median()
        ),
        "mean_popularity_share": (
            top["streamer_popularity_share"].mean()
        ),
    }


popularity_bias = pd.DataFrame(
    [
        topk_popularity(analysis, col, k=10)
        for col in SCORE_COLUMNS
    ]
)

popularity_bias

## 7. Inspect one user across all models

In [ ]:
example_user = pred["user_id"].iloc[0]

example = analysis[
    analysis["user_id"] == example_user
].copy()

display(
    example.sort_values(
        "score_neural",
        ascending=False,
    )[
        [
            "streamer",
            "label",
            "seen_before",
            "score_popularity",
            "score_logreg",
            "score_hgb",
            "score_neural",
        ]
    ].head(20)
)

## 8. Save evaluation tables

In [ ]:
comparison.to_csv(
    ARTIFACT_DIR / "ranking_model_comparison.csv",
    index=False,
)

popularity_bias.to_csv(
    ARTIFACT_DIR / "ranking_popularity_bias.csv",
    index=False,
)

print("Saved evaluation tables.")

## Important evaluation caveat

The ranking metrics above are calculated on the sampled candidate sets built in Notebook 02.

That evaluates **ranker quality given candidates**.

Notebook 05 separately measures **candidate-generation recall**. In the production system we will combine the two stages end-to-end.